# 05 - EKF on MuJoCo data: gains, noise models and filter variants

Collects a run on the MuJoCo robot (or loads `run_mujoco.npz` if it exists in this session; collection takes about 6 minutes on Colab), then replays it through the EKF variants. The waypoint follower reads the true pose to generate motion for the log and is not the project's controller; the filters never see the truth.

1. Odometry gains: nominal, one single-point calibration (one straight run and one spin), a **gain table** measured on arcs (forward speed and yaw rate together) in an obstacle-free room, and gains estimated online (variant B).
2. Marker noise: the isotropic model and an **anisotropic model** (depth error along the line of sight dominates). The anisotropic coefficients were read off a run like this one, so treat those rows as optimistic; see `docs/findings.md` (stages 5 to 7) for the diagnosis and the improved variants.
3. The odometry process-noise scale at 1, 2 and 4 times its default, and a 99.9% gate on the marker innovation.

The local script `experiments/mujoco_ekf.py` does the same without Colab.

In [ ]:
!pip install -q mujoco opencv-contrib-python
!test -d robot || git clone -q https://github.com/hongjiezheng-1-hue/robot.git
!cd robot && git pull -q
!test -d robotis_mujoco_menagerie || git clone -q --depth 1 --filter=blob:none --sparse https://github.com/ROBOTIS-GIT/robotis_mujoco_menagerie.git
!cd robotis_mujoco_menagerie && git sparse-checkout set robotis_tb3

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
import sys, time
sys.path.insert(0, "robot")
sys.path.insert(0, "robot/experiments")
import numpy as np
import matplotlib.pyplot as plt
from sim.scene import SceneConfig, write_scene
from sim.mujoco_run import RunConfig, collect_run, calibrate_gains, calibrate_gain_table, min_clearance, pick_texture_type
from drivers.mujoco_tb3 import MujocoTB3Driver
from estimation.ekf import marker_measurement_cov_measured
from replay import (replay, metrics, print_table, load_run, inflated, odometry_ratio_by_speed,
                    marker_residual_by_distance, CHI2_3_95, CHI2_3_999)

ROBOT_DIR = "robotis_mujoco_menagerie/robotis_tb3"
scene_cfg, run_cfg = pick_texture_type(SceneConfig(), ROBOT_DIR), RunConfig()
drv = MujocoTB3Driver(write_scene(scene_cfg, ROBOT_DIR))

if os.path.exists("run_mujoco.npz"):
    run = load_run("run_mujoco.npz")
    print("loaded run_mujoco.npz")
else:
    run = collect_run(drv, scene_cfg, run_cfg)
    np.savez("run_mujoco.npz", **run)
    print("collected a new run")
print(f"{len(run['truth'])} samples, marker pose available in {100 * np.mean(np.all(np.isfinite(run['marker']), axis=1)):.0f}% of samples")
print(f"smallest clearance of the robot centre to obstacles, station and walls: {min_clearance(run['truth'], scene_cfg):.2f} m")

In [ ]:
s_v_cal, s_w_cal = calibrate_gains(drv, run_cfg)
print(f"single-point calibration: s_v = {s_v_cal:.3f}, s_w = {s_w_cal:.3f}")

calib_cfg = SceneConfig(room_x=8.0, room_y=8.0, obstacles=(), station_x=3.5,
                        marker_texture_type=scene_cfg.marker_texture_type)   # obstacle-free room for arc calibration
calib_drv = MujocoTB3Driver(write_scene(calib_cfg, ROBOT_DIR, scene_name="scene_calib.xml"))
t0 = time.time()
gain_fn, table = calibrate_gain_table(calib_drv, run_cfg)
print(f"gain table measured in {time.time() - t0:.0f} s")
print("s_v (rows: nominal speed [m/s], columns: nominal yaw rate [rad/s])")
print("          " + "  ".join(f"{w:6.2f}" for w in table["yaw_rates"]))
for v, row in zip(table["speeds"], table["s_v"]):
    print(f"  {v:6.2f}  " + "  ".join(f"{x:6.3f}" for x in row))
print("s_w")
for v, row in zip(table["speeds"], table["s_w"]):
    print(f"  {v:6.2f}  " + "  ".join(f"{x:6.3f}" for x in row))

In [ ]:
aniso = marker_measurement_cov_measured
cal = (s_v_cal, s_w_cal)
variants = [
    ("A0 nominal gains", dict(gains=(1.0, 1.0))),
    ("A single-point", dict(gains=cal)),
    ("A single-point + gate", dict(gains=cal, gate=CHI2_3_999)),
    ("A gain table", dict(gain_fn=gain_fn)),
    ("A table + aniso R", dict(gain_fn=gain_fn, meas_cov=aniso)),
    ("A table + aniso R, Q x2", dict(gain_fn=gain_fn, meas_cov=aniso, noise=inflated(2))),
    ("A table + aniso R, Q x4", dict(gain_fn=gain_fn, meas_cov=aniso, noise=inflated(4))),
    ("B", dict(estimate=True)),
    ("B + gate", dict(estimate=True, gate=CHI2_3_999)),
    ("B + aniso R", dict(estimate=True, meas_cov=aniso)),
    ("B + aniso R, Q x2", dict(estimate=True, meas_cov=aniso, noise=inflated(2))),
    ("B + aniso R, Q x4", dict(estimate=True, meas_cov=aniso, noise=inflated(4))),
]
rows, results = [], {}
for name, kw in variants:
    errs, nees, hist, rejected = replay(run, **kw)
    rows.append((name, metrics(errs, nees), rejected))
    results[name] = (errs, nees, hist)
print_table(rows)
h = results["B + aniso R, Q x2"][2]
print(f"B + aniso R, Q x2 final gains: s_v = {h[-1, 0]:.3f}, s_w = {h[-1, 1]:.3f}")

In [ ]:
aniso = marker_measurement_cov_measured
cal = (s_v_cal, s_w_cal)
variants = [
    ("A single-point", dict(gains=cal)),
    ("A gain table", dict(gain_fn=gain_fn)),
    ("A table + aniso R", dict(gain_fn=gain_fn, meas_cov=aniso)),
    ("A table + aniso R, Q x2", dict(gain_fn=gain_fn, meas_cov=aniso, noise=inflated(2))),
    ("A table + aniso R, Q x4", dict(gain_fn=gain_fn, meas_cov=aniso, noise=inflated(4))),
    ("B", dict(estimate=True)),
    ("B + aniso R", dict(estimate=True, meas_cov=aniso)),
    ("B + aniso R, Q x2", dict(estimate=True, meas_cov=aniso, noise=inflated(2))),
    ("B + aniso R, Q x4", dict(estimate=True, meas_cov=aniso, noise=inflated(4))),
]
rows, results = [], {}
for name, kw in variants:
    errs, nees, hist, rejected = replay(run, **kw)
    rows.append((name, metrics(errs, nees), rejected))
    results[name] = (errs, nees, hist)
print_table(rows)
h = results["B + aniso R, Q x2"][2]
print(f"B + aniso R, Q x2 final gains: s_v = {h[-1, 0]:.3f}, s_w = {h[-1, 1]:.3f}")

In [ ]:
t = np.arange(1, len(run["truth"]) + 1) * run["dt"]
show = ["A single-point", "A table + aniso R, Q x2", "B", "B + aniso R, Q x2"]
fig, axs = plt.subplots(2, 1, figsize=(10, 8))
for name in show:
    errs, nees, _ = results[name]
    axs[0].plot(t, 1000 * np.linalg.norm(errs[:, :2], axis=1), lw=0.9, label=name)
    axs[1].plot(t, nees, lw=0.7, label=name)
axs[0].set_yscale("log"); axs[0].set_ylabel("position error [mm]"); axs[0].set_xlabel("time [s]")
axs[0].set_title("Position error on the MuJoCo run"); axs[0].legend(fontsize=8); axs[0].grid(True, which="both", alpha=0.3)
axs[1].set_yscale("log"); axs[1].axhline(CHI2_3_95, c="k", ls="--", lw=0.8, label="95% chi-square bound")
axs[1].set_ylabel("NEES"); axs[1].set_xlabel("time [s]"); axs[1].legend(fontsize=8); axs[1].grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.savefig("ekf_mujoco_improved.png", dpi=150); plt.show()